# 02. 라벨 구간 정렬

이 노트북은 HeatGrid Agent의 ML 파트에서 사용할 라벨 구간 후보를 정리한다.

목표는 고장 여부를 확정하는 것이 아니다.
고장 신고 전 위험 가능성, lead time, 정상 대비 이상 패턴을 학습할 수 있도록 fault / normal / disturbance 이벤트를 실제 기계실 운영 시계열 시간축에 맞춰 검증한다.

확인 결과는 `03_preprocess_windows.ipynb`에서 학습/평가용 시계열 윈도우를 만들 때 사용한다.


In [1]:
from pathlib import Path
import re

import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", 160)
pd.set_option("display.width", 180)


def find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "data" / "raw_data" / "predist_v2").exists():
            return candidate
    return start


ROOT_DIR = find_project_root(Path.cwd().resolve())
DATA_ROOT = ROOT_DIR / "data" / "raw_data" / "predist_v2"
OUTPUT_DIR = ROOT_DIR / "data" / "processed" / "label_alignment"

DEFAULT_FAULT_LOOKBACK = pd.Timedelta(days=3)

MANUFACTURERS = {
    "manufacturer 1": DATA_ROOT / "manufacturer 1",
    "manufacturer 2": DATA_ROOT / "manufacturer 2",
}


def read_semicolon_csv(path: Path, usecols=None) -> pd.DataFrame:
    return pd.read_csv(path, sep=";", encoding="utf-8-sig", low_memory=False, usecols=usecols)


def parse_substation_id(path: Path) -> int:
    match = re.search(r"substation_(\d+)\.csv$", path.name)
    if not match:
        raise ValueError(f"substation id를 파일명에서 찾을 수 없습니다: {path}")
    return int(match.group(1))


def to_datetime(series: pd.Series) -> pd.Series:
    return pd.to_datetime(series, errors="coerce")


def normalize_window(start, end):
    start = pd.to_datetime(start, errors="coerce")
    end = pd.to_datetime(end, errors="coerce")
    if pd.isna(start) and pd.isna(end):
        return pd.NaT, pd.NaT
    if pd.isna(start):
        start = end
    if pd.isna(end):
        end = start
    if start > end:
        start, end = end, start
    return start, end



def build_fault_window(row: pd.Series, lookback=DEFAULT_FAULT_LOOKBACK):
    report_date = row["Report date"]
    anomaly_start = row["Possible anomaly start"]
    anomaly_end = row["Possible anomaly end"]

    if pd.notna(report_date):
        end = report_date
    elif pd.notna(anomaly_end):
        end = anomaly_end
    else:
        end = anomaly_start

    if pd.notna(anomaly_start) and pd.notna(end) and anomaly_start < end:
        start = anomaly_start
    elif pd.notna(end):
        start = end - lookback
    else:
        start = pd.NaT

    return normalize_window(start, end)


## 1. 운영 데이터 시간 범위 확인

기계실별 운영 데이터가 언제부터 언제까지 존재하는지 확인한다.

라벨 구간이 운영 데이터 범위 밖에 있으면 ML 학습에 사용할 수 없다.
따라서 이 단계는 학습 가능한 기계실과 시간 구간을 선별하는 품질 검사다.


In [2]:
def build_operational_coverage(manufacturer: str, root: Path) -> pd.DataFrame:
    rows = []
    for path in sorted((root / "operational_data").glob("substation_*.csv")):
        timestamps = to_datetime(read_semicolon_csv(path, usecols=["timestamp"])["timestamp"])
        rows.append({
            "manufacturer": manufacturer,
            "substation_id": parse_substation_id(path),
            "file": path.name,
            "row_count": len(timestamps),
            "timestamp_min": timestamps.min(),
            "timestamp_max": timestamps.max(),
            "invalid_timestamp_rows": int(timestamps.isna().sum()),
        })
    return pd.DataFrame(rows)


coverage = pd.concat(
    [build_operational_coverage(name, root) for name, root in MANUFACTURERS.items()],
    ignore_index=True,
)

display(coverage.head())
display(
    coverage.groupby("manufacturer")
    .agg(
        file_count=("file", "count"),
        min_rows=("row_count", "min"),
        median_rows=("row_count", "median"),
        max_rows=("row_count", "max"),
        min_timestamp=("timestamp_min", "min"),
        max_timestamp=("timestamp_max", "max"),
    )
    .reset_index()
)


,manufacturer,substation_id,file,row_count,timestamp_min,timestamp_max,invalid_timestamp_rows
0,manufacturer 1,1,substation_1.csv,112164,2018-06-10 00:40:00,2020-07-28 23:50:00,0
1,manufacturer 1,10,substation_10.csv,150505,2012-03-28 08:00:00,2020-07-27 18:10:00,0
2,manufacturer 1,11,substation_11.csv,329685,2013-12-05 21:30:00,2020-07-28 23:50:00,0
3,manufacturer 1,12,substation_12.csv,300557,2013-11-30 20:30:00,2020-07-28 23:50:00,0
4,manufacturer 1,13,substation_13.csv,283544,2015-03-08 01:10:00,2020-07-28 23:50:00,0


,manufacturer,file_count,min_rows,median_rows,max_rows,min_timestamp,max_timestamp
0,manufacturer 1,35,34056,229972.0,347562,2012-03-28 08:00:00,2020-07-28 23:50:00
1,manufacturer 2,58,4844,26364.0,236084,2015-10-13 12:39:01,2020-04-02 00:00:00


## 2. 라벨 파일 읽기

fault, normal, disturbance 이벤트를 모두 시간 정보로 변환한다.

HeatGrid Agent 관점에서 각 파일의 의미는 아래와 같다.

- fault: 고장 신고 전 위험 구간 후보
- normal: 정상 패턴 학습 기준
- disturbance: 정비/작업으로 인한 센서 변화 해석 근거


In [3]:
def load_label_tables(manufacturer: str, root: Path) -> dict[str, pd.DataFrame]:
    faults = read_semicolon_csv(root / "faults.csv")
    normal_events = read_semicolon_csv(root / "normal_events.csv")
    disturbances = read_semicolon_csv(root / "disturbances.csv")

    for column in [
        "Report date",
        "Possible anomaly start",
        "Possible anomaly end",
        "Training start",
        "Training end",
    ]:
        faults[column] = to_datetime(faults[column])

    for column in ["Event start", "Event end", "Training start", "Training end"]:
        normal_events[column] = to_datetime(normal_events[column])

    disturbances["Event start"] = to_datetime(disturbances["Event start"])

    faults.insert(0, "manufacturer", manufacturer)
    normal_events.insert(0, "manufacturer", manufacturer)
    disturbances.insert(0, "manufacturer", manufacturer)

    return {
        "faults": faults,
        "normal_events": normal_events,
        "disturbances": disturbances,
    }


label_tables = {
    name: load_label_tables(name, root)
    for name, root in MANUFACTURERS.items()
}

faults = pd.concat([tables["faults"] for tables in label_tables.values()], ignore_index=True)
normal_events = pd.concat([tables["normal_events"] for tables in label_tables.values()], ignore_index=True)
disturbances = pd.concat([tables["disturbances"] for tables in label_tables.values()], ignore_index=True)

display(pd.DataFrame([
    {"table": "faults", "rows": len(faults)},
    {"table": "normal_events", "rows": len(normal_events)},
    {"table": "disturbances", "rows": len(disturbances)},
]))
display(faults[["manufacturer", "Event ID", "substation ID", "Report date", "Possible anomaly start", "Possible anomaly end", "Fault label"]].head())


,table,rows
0,faults,73
1,normal_events,65
2,disturbances,328


,manufacturer,Event ID,substation ID,Report date,Possible anomaly start,Possible anomaly end,Fault label
0,manufacturer 1,1,10,2014-05-04 14:44:00,2014-05-03 16:00:00,2014-05-05 04:00:00,Motorised control valve (primary side): Actuat...
1,manufacturer 1,3,12,2015-12-01 10:56:00,2015-11-29 12:00:00,2015-12-02 10:56:00,Control unit: Incorrect parameterisation
2,manufacturer 1,5,11,2018-11-23 08:30:00,NaT,2018-11-26 09:56:59,Failure of the heating circuit pump
3,manufacturer 1,6,21,2016-12-06 13:12:00,NaT,2016-12-07 13:12:00,Control unit: Incorrect parameterisation
4,manufacturer 1,7,26,2020-06-13 10:38:00,2020-06-12 12:00:00,2020-06-14 10:38:00,Incorrect setting of the differential pressure...


## 3. 구간 정렬 기준

라벨 구간이 운영 데이터 범위와 겹치는지 확인한다.

`is_usable`은 이 구간을 ML 학습/평가 후보로 써도 되는지 판단하는 품질 플래그다.
Agent에게 넘길 위험 판단 근거를 만들기 전, 먼저 데이터가 실제로 존재하는 구간만 남겨야 한다.

issue 기준:

- `usable`: 운영 데이터와 라벨 구간이 겹치고 실제 row가 있음
- `no_operational_file`: 해당 기계실 운영 CSV가 없음
- `missing_window`: 라벨의 시작/종료 시각을 만들 수 없음
- `out_of_range`: 운영 데이터 범위와 라벨 구간이 겹치지 않음
- `no_rows_in_window`: 시간 범위는 겹치지만 실제 timestamp row가 없음


In [4]:
timestamp_cache = {}


def get_timestamps(manufacturer: str, substation_id: int) -> pd.Series:
    key = (manufacturer, int(substation_id))
    if key not in timestamp_cache:
        root = MANUFACTURERS[manufacturer]
        path = root / "operational_data" / f"substation_{int(substation_id)}.csv"
        if not path.exists():
            timestamp_cache[key] = None
        else:
            timestamp_cache[key] = to_datetime(read_semicolon_csv(path, usecols=["timestamp"])["timestamp"])
    return timestamp_cache[key]


def count_rows_in_window(manufacturer: str, substation_id: int, start, end) -> int:
    timestamps = get_timestamps(manufacturer, substation_id)
    if timestamps is None or pd.isna(start) or pd.isna(end):
        return 0
    return int(((timestamps >= start) & (timestamps <= end)).sum())


def align_window(manufacturer: str, substation_id: int, window_start, window_end) -> dict:
    start, end = normalize_window(window_start, window_end)
    matched = coverage[
        (coverage["manufacturer"] == manufacturer)
        & (coverage["substation_id"] == int(substation_id))
    ]

    if matched.empty:
        return {
            "window_start": start,
            "window_end": end,
            "data_start": pd.NaT,
            "data_end": pd.NaT,
            "available_rows": 0,
            "is_usable": False,
            "issue": "no_operational_file",
        }

    data = matched.iloc[0]
    data_start = data["timestamp_min"]
    data_end = data["timestamp_max"]

    if pd.isna(start) or pd.isna(end):
        return {
            "window_start": start,
            "window_end": end,
            "data_start": data_start,
            "data_end": data_end,
            "available_rows": 0,
            "is_usable": False,
            "issue": "missing_window",
        }

    overlap_start = max(start, data_start)
    overlap_end = min(end, data_end)
    if overlap_start > overlap_end:
        return {
            "window_start": start,
            "window_end": end,
            "data_start": data_start,
            "data_end": data_end,
            "available_rows": 0,
            "is_usable": False,
            "issue": "out_of_range",
        }

    available_rows = count_rows_in_window(manufacturer, substation_id, overlap_start, overlap_end)
    return {
        "window_start": start,
        "window_end": end,
        "data_start": data_start,
        "data_end": data_end,
        "overlap_start": overlap_start,
        "overlap_end": overlap_end,
        "available_rows": available_rows,
        "is_usable": available_rows > 0,
        "issue": "usable" if available_rows > 0 else "no_rows_in_window",
    }



def align_point_in_range(manufacturer: str, substation_id: int, event_time) -> dict:
    event_time = pd.to_datetime(event_time, errors="coerce")
    matched = coverage[
        (coverage["manufacturer"] == manufacturer)
        & (coverage["substation_id"] == int(substation_id))
    ]

    if matched.empty:
        return {
            "window_start": event_time,
            "window_end": event_time,
            "data_start": pd.NaT,
            "data_end": pd.NaT,
            "available_rows": pd.NA,
            "is_usable": False,
            "issue": "no_operational_file",
        }

    data = matched.iloc[0]
    data_start = data["timestamp_min"]
    data_end = data["timestamp_max"]

    if pd.isna(event_time):
        return {
            "window_start": event_time,
            "window_end": event_time,
            "data_start": data_start,
            "data_end": data_end,
            "available_rows": pd.NA,
            "is_usable": False,
            "issue": "missing_window",
        }

    is_in_range = data_start <= event_time <= data_end
    return {
        "window_start": event_time,
        "window_end": event_time,
        "data_start": data_start,
        "data_end": data_end,
        "overlap_start": event_time if is_in_range else pd.NaT,
        "overlap_end": event_time if is_in_range else pd.NaT,
        "available_rows": pd.NA,
        "is_usable": bool(is_in_range),
        "issue": "usable" if is_in_range else "out_of_range",
    }


## 4. fault 구간 정렬

fault 구간은 고장 신고 전 위험 가능성을 학습하기 위한 핵심 기준이다.

`Possible anomaly start/end`를 우선 사용해 고장 신고 전 이상 가능 구간을 만든다.
이 구간은 이후 모델이 위험 점수, lead time, 고장 라벨 후보를 계산하는 기준이 된다.


In [5]:
fault_rows = []
for _, row in faults.iterrows():
    fault_start, fault_end = build_fault_window(row)
    aligned = align_window(
        row["manufacturer"],
        row["substation ID"],
        fault_start,
        fault_end,
    )
    fault_rows.append({
        "manufacturer": row["manufacturer"],
        "event_id": row["Event ID"],
        "substation_id": row["substation ID"],
        "label_type": "fault",
        "fault_label": row["Fault label"],
        "problem": row["Problem EN"],
        "report_date": row["Report date"],
        "used_fallback_window": pd.isna(row["Possible anomaly start"]),
        **aligned,
    })

fault_alignment = pd.DataFrame(fault_rows)
display(fault_alignment.head())
display(fault_alignment["issue"].value_counts().rename_axis("issue").reset_index(name="count"))


,manufacturer,event_id,substation_id,label_type,fault_label,problem,report_date,used_fallback_window,window_start,window_end,data_start,data_end,overlap_start,overlap_end,available_rows,is_usable,issue
0,manufacturer 1,1,10,fault,Motorised control valve (primary side): Actuat...,no DHW,2014-05-04 14:44:00,False,2014-05-03 16:00:00,2014-05-04 14:44:00,2012-03-28 08:00:00,2020-07-27 18:10:00,2014-05-03 16:00:00,2014-05-04 14:44:00,137,True,usable
1,manufacturer 1,3,12,fault,Control unit: Incorrect parameterisation,no heat,2015-12-01 10:56:00,False,2015-11-29 12:00:00,2015-12-01 10:56:00,2013-11-30 20:30:00,2020-07-28 23:50:00,2015-11-29 12:00:00,2015-12-01 10:56:00,282,True,usable
2,manufacturer 1,5,11,fault,Failure of the heating circuit pump,no heat,2018-11-23 08:30:00,True,2018-11-20 08:30:00,2018-11-23 08:30:00,2013-12-05 21:30:00,2020-07-28 23:50:00,2018-11-20 08:30:00,2018-11-23 08:30:00,433,True,usable
3,manufacturer 1,6,21,fault,Control unit: Incorrect parameterisation,not enough heat,2016-12-06 13:12:00,True,2016-12-03 13:12:00,2016-12-06 13:12:00,2015-02-23 19:40:00,2020-07-28 23:50:00,2016-12-03 13:12:00,2016-12-06 13:12:00,432,True,usable
4,manufacturer 1,7,26,fault,Incorrect setting of the differential pressure...,no DHW,2020-06-13 10:38:00,False,2020-06-12 12:00:00,2020-06-13 10:38:00,2018-10-18 13:00:00,2020-07-28 23:50:00,2020-06-12 12:00:00,2020-06-13 10:38:00,136,True,usable


,issue,count
0,usable,73


## 5. normal 구간 정렬

normal 구간은 정상 운전 패턴의 기준이다.

ML은 정상 패턴을 알아야 fault 구간과 다른 변화를 이상 신호로 볼 수 있다.
따라서 normal 구간이 운영 데이터와 실제로 맞는지 먼저 확인한다.


In [6]:
normal_rows = []
for _, row in normal_events.iterrows():
    aligned = align_window(
        row["manufacturer"],
        row["substation ID"],
        row["Event start"],
        row["Event end"],
    )
    normal_rows.append({
        "manufacturer": row["manufacturer"],
        "event_id": row["Event ID"],
        "substation_id": row["substation ID"],
        "label_type": "normal",
        **aligned,
    })

normal_alignment = pd.DataFrame(normal_rows)
display(normal_alignment.head())
display(normal_alignment["issue"].value_counts().rename_axis("issue").reset_index(name="count"))


,manufacturer,event_id,substation_id,label_type,window_start,window_end,data_start,data_end,overlap_start,overlap_end,available_rows,is_usable,issue
0,manufacturer 1,2,9,normal,2019-01-02,2019-01-09,2015-02-15 23:40:00,2020-07-27 18:20:00,2019-01-02,2019-01-09,1009,True,usable
1,manufacturer 1,4,15,normal,2019-03-01,2019-03-08,2014-01-23 02:30:00,2020-07-28 23:50:00,2019-03-01,2019-03-08,1009,True,usable
2,manufacturer 1,8,6,normal,2020-06-02,2020-06-09,2019-03-13 09:10:00,2020-07-27 18:20:00,2020-06-02,2020-06-09,1009,True,usable
3,manufacturer 1,12,19,normal,2017-09-30,2017-10-07,2016-02-22 13:40:00,2020-07-28 23:50:00,2017-09-30,2017-10-07,1009,True,usable
4,manufacturer 1,14,26,normal,2019-12-08,2019-12-15,2018-10-18 13:00:00,2020-07-28 23:50:00,2019-12-08,2019-12-15,1009,True,usable


,issue,count
0,usable,65


## 6. disturbance 시점 확인

disturbance는 정비/작업 이력이다.

이 정보는 모델 학습 라벨이라기보다 Agent가 결과를 해석할 때 필요한 보조 근거다.
예를 들어 이상 패턴이 정비 직후에 나타났다면, Agent는 이를 점검 이력과 함께 설명할 수 있어야 한다.


In [7]:
disturbance_rows = []
for _, row in disturbances.iterrows():
    aligned = align_point_in_range(
        row["manufacturer"],
        row["substation ID"],
        row["Event start"],
    )
    disturbance_rows.append({
        "manufacturer": row["manufacturer"],
        "substation_id": row["substation ID"],
        "label_type": "disturbance",
        "disturbance_type": row["type"],
        "event_start": row["Event start"],
        **aligned,
    })

disturbance_alignment = pd.DataFrame(disturbance_rows)
display(disturbance_alignment.head())
display(disturbance_alignment["issue"].value_counts().rename_axis("issue").reset_index(name="count"))


,manufacturer,substation_id,label_type,disturbance_type,event_start,window_start,window_end,data_start,data_end,overlap_start,overlap_end,available_rows,is_usable,issue
0,manufacturer 1,26,disturbance,fault,2020-03-10 15:49:00,2020-03-10 15:49:00,2020-03-10 15:49:00,2018-10-18 13:00:00,2020-07-28 23:50:00,2020-03-10 15:49:00,2020-03-10 15:49:00,<NA>,True,usable
1,manufacturer 1,3,disturbance,activity,2015-11-24 00:00:00,2015-11-24 00:00:00,2015-11-24 00:00:00,2015-11-09 16:40:00,2020-07-28 23:50:00,2015-11-24 00:00:00,2015-11-24 00:00:00,<NA>,True,usable
2,manufacturer 1,3,disturbance,fault,2015-12-09 13:48:00,2015-12-09 13:48:00,2015-12-09 13:48:00,2015-11-09 16:40:00,2020-07-28 23:50:00,2015-12-09 13:48:00,2015-12-09 13:48:00,<NA>,True,usable
3,manufacturer 1,3,disturbance,activity,2016-02-29 00:00:00,2016-02-29 00:00:00,2016-02-29 00:00:00,2015-11-09 16:40:00,2020-07-28 23:50:00,2016-02-29 00:00:00,2016-02-29 00:00:00,<NA>,True,usable
4,manufacturer 1,3,disturbance,fault,2016-10-31 14:18:00,2016-10-31 14:18:00,2016-10-31 14:18:00,2015-11-09 16:40:00,2020-07-28 23:50:00,2016-10-31 14:18:00,2016-10-31 14:18:00,<NA>,True,usable


,issue,count
0,usable,311
1,no_operational_file,16
2,out_of_range,1


## 7. 결과 저장

정렬 결과를 `data/processed/label_alignment/` 아래에 저장한다.

다음 단계에서는 `is_usable = True`인 구간을 우선 사용해 학습/평가용 윈도우를 만든다.
이후 ML 산출물은 Agent가 사용할 수 있도록 기계실 ID, 시간 구간, 위험 가능성, lead time, 주요 센서 후보와 함께 정리된다.


In [8]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

coverage.to_csv(OUTPUT_DIR / "operational_coverage.csv", index=False, encoding="utf-8-sig")
fault_alignment.to_csv(OUTPUT_DIR / "fault_alignment.csv", index=False, encoding="utf-8-sig")
normal_alignment.to_csv(OUTPUT_DIR / "normal_alignment.csv", index=False, encoding="utf-8-sig")
disturbance_alignment.to_csv(OUTPUT_DIR / "disturbance_alignment.csv", index=False, encoding="utf-8-sig")

print(f"saved: {OUTPUT_DIR}")


saved: C:\Users\user\Heat_Grid_Agent\data\processed\label_alignment
